## Process Orders data

1. Ingest the data into the data lakehouse - bronze_orders

1. Ingest the data into the data lakehouse - bronze_order
2. Perform data quality checks and transform the data as required - silver_order_clean
3. Explode tge items arrat from the order object - silver_orders

1. Ingest the data into the data lakehouse - bronze_order

In [0]:
CREATE OR REFRESH STREAMING TABLE bronze_orders
COMMENT "Raw orders data ingested fro the source system"
TBLPROPERTIES ("quality" = "bronze")
AS
SELECT *,
        _metadata.file_path AS input_file_path,
        CURRENT_TIMESTAMP AS ingestion_timestamp
    FROM cloud_files(
                        '/Volumes/circuitbox/landing/operational_data/orders/',
                        'json',
                        map("cloudFiles.inferColumnTypes", "true")
                    );

2. Perform data quality checks and transform the data as required - silver_order_clean

In [0]:
CREATE OR REFRESH STREAMING TABLE silver_orders_clean(
  CONSTRAINT valid_customer_id EXPECT (customer_id IS NOT NULL) ON VIOLATION FAIL UPDATE,
  CONSTRAINT valid_order_id EXPECT (order_ID IS NOT NULL) ON VIOLATION FAIL UPDATE,
  CONSTRAINT valid_order_status EXPECT (order_status IN('Credit Card', 'Bank Transfer', 'Paypal'))
)
COMMENT "Cleaned orders data"
AS
SELECT order_id,
       customer_id,
       CAST(order_timestamp AS TIMESTAMP) AS order_timestamp,
       payment_method,
       items,
       order_status 
FROM STREAM(LIVE.bronze_orders);

3. Explode the items arrat from the order object - silver_orders

In [0]:
CREATE STREAMING TABLE silver_orders
AS
SELECT order_id,
       customer_id,
       order_timestamp,
       payment_method,
       order_status,
       item.item_id,
       item.name AS item_name,
       item.price AS item_price,
       item.quantity AS item_quantity,
       item.category AS item_category
FROM (SELECT order_id,
             customer_id,
             order_timestamp,
             payment_method,
             order_status,
             explode(items) AS item
      FROM STREAM(LIVE.silver_orders_clean));
             